In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from src.data_loader import get_prices, get_log_returns, get_stats

# Télécharger Apple, Google, Microsoft (2 ans)
tickers = ["AAPL", "GOOGL", "MSFT"]
prices = get_prices(tickers, start="2023-01-01", end="2025-01-01")
returns = get_log_returns(prices)
mu, sigma = get_stats(returns)

print("Prix actuels:\n", prices.tail(3))
print("\nRendements annualisés:\n", mu.round(4))
print("\nVolatilités annualisées:\n", sigma.round(4))

[*********************100%***********************]  3 of 3 completed

Prix actuels:
 Ticker            AAPL       GOOGL        MSFT
Date                                          
2024-12-27  253.967361  191.758392  425.482483
2024-12-30  250.598892  190.246307  419.849335
2024-12-31  248.830200  188.316376  416.558411

Rendements annualisés:
 Ticker
AAPL     0.3545
GOOGL    0.3808
MSFT     0.2923
dtype: float64

Volatilités annualisées:
 Ticker
AAPL     0.2129
GOOGL    0.2921
MSFT     0.2266
dtype: float64


In [2]:
from src.simulation import simulate_gbm
from src.visualization import plot_simulations

# Simuler Apple sur 1 an, 500 trajectoires
S0_aapl = float(prices["AAPL"].iloc[-1])
paths = simulate_gbm(S0=S0_aapl, mu=float(mu["AAPL"]),
                     sigma=float(sigma["AAPL"]), T=252, n_simulations=500)

plot_simulations(paths, ticker="AAPL")
print(f"Prix final moyen simulé : ${paths[-1].mean():.2f}")
print(f"Intervalle 95% : [${np.percentile(paths[-1], 2.5):.2f}, ${np.percentile(paths[-1], 97.5):.2f}]")

Prix final moyen simulé : $354.82
Intervalle 95% : [$235.58, $527.90]


In [3]:
from src.risk import var_historique, var_monte_carlo, cvar, sharpe_ratio, max_drawdown
from src.visualization import plot_var_distribution

aapl_returns = returns["AAPL"].values

var_95 = var_historique(aapl_returns, alpha=0.05)
var_99 = var_historique(aapl_returns, alpha=0.01)
cvar_95 = cvar(aapl_returns, alpha=0.05)
sr = sharpe_ratio(aapl_returns)
mdd = max_drawdown(prices["AAPL"].values)

print(f"VaR 95%  : {var_95:.4f} ({var_95*100:.2f}%)")
print(f"VaR 99%  : {var_99:.4f} ({var_99*100:.2f}%)")
print(f"CVaR 95% : {cvar_95:.4f} ({cvar_95*100:.2f}%)")
print(f"Sharpe   : {sr:.4f}")
print(f"Max Drawdown : {mdd*100:.2f}%")

plot_var_distribution(aapl_returns, var_95)

VaR 95%  : -0.0203 (-2.03%)
VaR 99%  : -0.0297 (-2.97%)
CVaR 95% : -0.0283 (-2.83%)
Sharpe   : 1.5723
Max Drawdown : -16.61%


In [4]:
from src.optimization import max_sharpe, min_variance, efficient_frontier
from src.visualization import plot_frontier

mu_vec = mu.values
cov_matrix = returns.cov().values * 252

w_sharpe = max_sharpe(mu_vec, cov_matrix)
w_minvar = min_variance(mu_vec, cov_matrix)
target_ret, frontier_vol = efficient_frontier(mu_vec, cov_matrix)

print("Portefeuille Sharpe Maximum :")
for t, w in zip(tickers, w_sharpe):
    print(f"  {t}: {w*100:.1f}%")

print("\nPortefeuille Variance Minimale :")
for t, w in zip(tickers, w_minvar):
    print(f"  {t}: {w*100:.1f}%")

plot_frontier(target_ret, frontier_vol)

Portefeuille Sharpe Maximum :
  AAPL: 62.6%
  GOOGL: 21.4%
  MSFT: 16.0%

Portefeuille Variance Minimale :
  AAPL: 52.5%
  GOOGL: 9.1%
  MSFT: 38.4%
